In [25]:
import os
import json
import requests


transcripts = {}
for i in range(8):
    transcripts[i+1] = {}


def get_transcript(call_id):
    headers = { 'Authorization': os.getenv('BLAND_AUTH_TOKEN') }
    response = requests.request("GET", f"https://api.bland.ai/v1/calls/{call_id}", headers=headers, verify=False)
    response_dict = json.loads(response.text)

    return response_dict["concatenated_transcript"]


def set_transcripts(participant_id, call_ids):
    for i, call_id in enumerate(call_ids, 1):
        transcripts[participant_id][f'{i}_{call_id}'] = get_transcript(call_id)


In [31]:
participants = {
    1:
        [
            'm', 'nonnative', 
            ["236c75ef-31f0-41a5-9e04-6699c121a94e", "1ff0a420-77ba-46b1-9873-f8b29c05aa41", "f311bb3e-75a7-4300-95f5-a7afb8fd29a5", 
            "0e33d383-5e2c-47b0-ace4-a3efe5f966cb", "94441d4f-f4e1-4156-9a7e-342fb123d166"]
        ],
    2:
        [
            'm', 'native', 
            ["0a6bc04e-2b15-4e12-95ea-9e590d377e95", "fed90af9-0475-46aa-b7f5-1d332a6f6b23", "945ad965-8cc1-4f7a-9a4e-bbd458aa587c", 
            "f4b0d4bf-62ef-4716-8488-ab062501f81e", "f82df6f2-d6f4-44de-85a0-c5359baf8db7"]
        ],
    3:
        [
            'f', 'nonnative', 
            ['46b05daf-6232-4aca-88ce-b284df24e67e', 'c2edfc21-c4e6-4f3a-8d9f-bc2f895f7123', '6b39b9ad-605d-4b4c-91fa-8cc90da45943',
            'e7c63a0f-41c0-48c7-a851-721e90f0635a', '5aeae7d7-fa9f-4773-9393-434cbdb612d8']
        ],
    4:
        [
            'm', 'native', 
            ["dbfb95a3-b299-482b-89e4-969242446167", "34b3ab14-4207-4759-a159-08238255c7db", "8d6fedcc-c02b-47c3-b21e-f033f494b80b", 
             "1c1e55b6-61b2-4b64-a297-f3625d562a19", "017f2502-46bb-44a7-80d3-0d9a9ed04afe"]
        ],
    5:
        [
            'm', 'native', 
            ["50c4c855-4728-4b01-ba47-0d666adf4597", "8c0aa1a9-4bf7-40fc-856c-d43ef03d3ecd", "2a34129c-7ab4-494c-b9a0-40013fc33551",
             "3cccbae0-a68c-4984-95df-60395d43bcfb", "c20805bf-c205-4eda-a898-3088140e12b6"]
        ]
}


In [32]:
for id, [sex, native, call_ids] in participants.items():
    if id != 5:
        continue
    set_transcripts(id, call_ids)

/Users/dobbinsnj/work/ai-agent-based-survey/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.bland.ai'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/dobbinsnj/work/ai-agent-based-survey/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.bland.ai'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/dobbinsnj/work/ai-agent-based-survey/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.bland.ai'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usa

In [33]:
for participant_id, values in transcripts.items():
    
    if participant_id != 5:
        continue

    sex, native, _ = participants[participant_id]

    for call_id, transcript in values.items():
        directory = os.path.join('./call-data', 'calls', f'{participant_id}_{sex}_{native}_speaker')
        if not os.path.exists(directory):
            os.mkdir(directory)

        if not os.path.exists(os.path.join(directory, f'call_{call_id}.txt')):
            with open(os.path.join(directory, f'call_{call_id}.txt'), 'w+', encoding='utf-8') as fout:
                fout.write(transcript)
            with open(os.path.join(directory, f'call_{call_id}_corrected.txt'), 'w+', encoding='utf-8') as fout:
                fout.write(transcript)

In [45]:
import os
import sacrebleu
from rouge_score import rouge_scorer


scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
data_path = os.path.join('./call-data', 'calls')
rouge = {}

def user_responses_only(text):
    output = []
    for line in text.split('\n'):
        if line.startswith('assistant:') or not line.strip():
            continue
        line = line.replace('user:','').replace('<Cannot be interrupted, user message ignored>', '').strip()
        output.append(line)

    return output

for dir in os.listdir(data_path):
    participant_id, sex, speaker, _ = dir.split('_')
    rouge[participant_id] = []
    transcripts = [x for x in os.listdir(os.path.join(data_path, dir)) if 'corrected' not in x]

    for transcript_path in transcripts:
        with open(os.path.join(data_path, dir, transcript_path), 'r', encoding='utf-8') as fin:
            original = user_responses_only(fin.read())
        with open(os.path.join(data_path, dir, transcript_path.replace('.txt', '_corrected.txt')), 'r', encoding='utf-8') as fin:
            corrected = user_responses_only(fin.read())

        assert len(original) == len(corrected), f"Lengths don't match! {dir}/{transcript_path} {len(original)} {len(corrected)}"

        original  = "\n".join(original)
        corrected = "\n".join(corrected)

        #score = sacrebleu.raw_corpus_bleu(original, [corrected], 0.0).score / 100
        score = scorer.score(original, corrected)
        rouge[participant_id].append(score['rougeL'].fmeasure)

rouge

{'3': [0.9953343701399688,
  0.9865908167411622,
  0.9884969325153373,
  0.9951377633711508,
  0.9908151549942595],
 '2': [1.0,
  0.9975990396158463,
  0.9925536574682435,
  0.9994321408290744,
  0.9889258028792912],
 '1': [0.9929836284664217,
  0.9897292250233426,
  0.9902097902097902,
  0.9955334987593052,
  0.9875444839857651],
 '5': [0.9941291585127201,
  0.9986168741355463,
  0.9951100244498777,
  0.9938461538461538,
  0.9960474308300395],
 '4': [0.9969525468001741,
  0.9918166939443536,
  0.9912931312479846,
  0.9958018471872376,
  0.9978417266187051]}

In [69]:
import os
import spacy
import jiwer
from statistics import mean


transforms = jiwer.Compose(
    [
        jiwer.ExpandCommonEnglishContractions(),
        jiwer.RemoveEmptyStrings(),
        jiwer.ToLowerCase(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
        jiwer.RemovePunctuation(),
        jiwer.ReduceToListOfListOfWords(),
    ]
)

#nlp = spacy.load('en_core_web_sm')
data_path = os.path.join('./call-data', 'calls')
token_accuracy = {}


for dir in os.listdir(data_path):
    participant_id, sex, speaker, _ = dir.split('_')
    token_accuracy[participant_id] = []
    transcripts = [x for x in os.listdir(os.path.join(data_path, dir)) if 'corrected' not in x]

    for transcript_path in transcripts:
        with open(os.path.join(data_path, dir, transcript_path), 'r', encoding='utf-8') as fin:
            original = user_responses_only(fin.read())
        with open(os.path.join(data_path, dir, transcript_path.replace('.txt', '_corrected.txt')), 'r', encoding='utf-8') as fin:
            corrected = user_responses_only(fin.read())

        assert len(original) == len(corrected), f"Lengths don't match! {dir}/{transcript_path} {len(original)} {len(corrected)}"

        for orig_line, gold_line in zip(original, corrected):
            #orig_tokens, gold_tokens = nlp(orig_line), nlp(gold_line)

            # https://cloud.google.com/speech-to-text/docs/speech-accuracy#:~:text=Speech%20accuracy%20can%20be%20measured,transcriptions%20in%20the%20entire%20set.
            # ---
            # Insertion Error (I): Words present in the hypothesis transcript that aren't present in the ground truth.
            # Substitution errors (S): Words that are present in both the hypothesis and ground truth but aren't transcribed correctly.
            # Deletion errors (D): Words that are missing from the hypothesis but present in the ground truth.

            if any(orig_line) and any(gold_line):
                word_error_rate = jiwer.wer(
                    orig_line,
                    gold_line,
                    truth_transform=transforms,
                    hypothesis_transform=transforms,
                )

                token_accuracy[participant_id].append([orig_line, gold_line, word_error_rate])

            #if any(orig_tokens):
            #    orig_tokens     = set([str(x) for x in orig_tokens])
            #    gold_tokens     = set([str(x) for x in gold_tokens])
            #    deletions       = len([x for x in orig_tokens if str(x) not in gold_tokens])
            #    insertions      = len([x for x in gold_tokens if str(x) not in orig_tokens])
            #    substitutions   = 0 # not considering substitions for now
            #    word_error_rate = (deletions + insertions + substitutions) / len(orig_tokens)
            #    #if len(orig_tokens) == deletions + insertions:
            #    #    accuracy = 0
            #    #else:
            #    #    accuracy = (len(orig_tokens) - deletions - insertions) / len(orig_tokens)
            #    token_accuracy[participant_id].append([len(orig_tokens), deletions + insertions + substitutions, word_error_rate])

for participant_id, values in sorted(token_accuracy.items()):
    mean_macro_wer = round(mean([v[2] for v in values]) * 100,1) # ie, per line
    #micro_wer      = round(sum([v[1] for v in values]) / sum([v[0] for v in values]) * 100,1) # ie, across all transcripts

    orig_transcripts = '\n'.join(orig_line for orig_line, _, _ in values)
    gold_transcripts = '\n'.join(gold_line for _, gold_line, _ in values)

    word_error_rate = jiwer.wer(
        orig_transcripts,
        gold_transcripts,
        truth_transform=transforms,
        hypothesis_transform=transforms,
    )

    print(f'{participant_id}: Per-line WER: {mean_macro_wer:>4}%, Micro-WER: {round(word_error_rate * 100,1)}%')



1: Per-line WER:  3.2%, Micro-WER: 1.3%
2: Per-line WER:  2.0%, Micro-WER: 0.7%
3: Per-line WER:  9.9%, Micro-WER: 1.4%
4: Per-line WER:  3.3%, Micro-WER: 0.7%
5: Per-line WER:  4.9%, Micro-WER: 0.6%
